In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score
from transformers import AutoTokenizer,TFAutoModelForSequenceClassification,pipeline

from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
df=pd.read_csv("all_kindle_review.csv")
df.head(5)

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
df=df[['rating','reviewText']].dropna()
df.isna().sum()

rating        0
reviewText    0
dtype: int64

In [4]:
df['rating'] = (df['rating'] > 3).astype(int)

In [5]:
df['rating'].value_counts()

rating
0    6000
1    6000
Name: count, dtype: int64

In [20]:
train_x,test_x,train_y,test_y=train_test_split(df['reviewText'],df['rating'],random_state=42,test_size=0.2,stratify=df['rating'])

In [28]:
train_x=train_x.reset_index(drop=True)
train_y=train_y.reset_index(drop=True)
test_x=test_x.reset_index(drop=True)
test_y=test_y.reset_index(drop=True)

In [ ]:
model=TFAutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english',from_pt=True)
tokenizer=AutoTokenizer.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFDistilBertForSequenceClassification.

All the weights of TFDistilBertForSequenceClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForSequenceClassification for predictions without further training.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [36]:
train_encodings = tokenizer(
    list(train_x),
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    list(test_x),
    truncation=True,
    padding=True,
    max_length=64
)

In [38]:
import tensorflow as tf

train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    list(train_y)
)).shuffle(1000).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    list(test_y)
)).batch(16)

In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002912CE49070>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 5c6c7773-619b-4918-8375-0ff13f1cfb39)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTrain

In [40]:
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

Model Fine Tuning

In [42]:
model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=1
)

600/600 [==============================] - 698s 1s/step - loss: 0.3689 - accuracy: 0.8388 - val_loss: 0.3699 - val_accuracy: 0.8379


In [45]:
classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    truncation=True
)


In [59]:
outputs = model.predict(test_dataset)
logits = outputs.logits

150/150 [==============================] - 41s 270ms/step


In [60]:
y_pred = np.argmax(logits, axis=1)

In [61]:
print("Accuracy:", accuracy_score(test_y, y_pred))
print("Precision:", precision_score(test_y, y_pred))

Accuracy: 0.8379166666666666
Precision: 0.7992619926199263
